Grupo 4 - 13/05/2026 - IA - Trabajo IA - Detección y Recorte de Caras

# Notebook 2: Detección y Recorte de Caras

En el Notebook 1 analizamos el dataset de fotos del Gobierno de La Rioja y comprobamos que el 82,2% de las imágenes contienen al menos una cara detectada con Haar Cascades. Sin embargo, como mencionamos, Haar Cascades es un detector rápido pero poco preciso que genera falsos positivos y pierde caras de perfil o con oclusiones,

En este notebook haremos la detección de verdad primero comparando tres detectores. Una vez elegido el mejor recortaremos y normalizaremos las caras de las imagenes del dataset.

## 0. Carga de librerías

Cargamos las librerías necesarias para la deteccion.

In [1]:
%%capture
!pip install cmake
!pip install dlib
!pip install opencv-contrib-python

In [2]:
import os
import cv2
import dlib
import numpy as np
import matplotlib.pyplot as plt
import random
import time
import concurrent.futures
import urllib.request
from pathlib import Path
from collections import defaultdict
%matplotlib inline

## 1. Carga del Dataset

Descargamos el dataset desde el mismo release de GitHub que usamos en el Notebook 1.

In [3]:
!wget DATASET_URL -O dados.zip
!unzip dados.zip

Se han truncado las últimas 5000 líneas del flujo de salida.
  inflating: 2/467124.jpg            
  inflating: 2/467130.jpg            
  inflating: 2/467131.jpg            
  inflating: 2/467132.jpg            
  inflating: 2/467135.jpg            
  inflating: 2/467136.jpg            
  inflating: 2/467138.jpg            
  inflating: 2/467139.jpg            
  inflating: 2/467140.jpg            
  inflating: 2/46715.jpg             
  inflating: 2/467156.jpg            
  inflating: 2/46716.jpg             
  inflating: 2/467161.jpg            
  inflating: 2/467163.jpg            
  inflating: 2/467164.jpg            
  inflating: 2/467169.jpg            
  inflating: 2/46717.jpg             
  inflating: 2/467170.jpg            
  inflating: 2/467173.jpg            
  inflating: 2/453565.jpg            
  inflating: 2/453569.jpg            
  inflating: 2/45357.jpg             
  inflating: 2/453572.jpg            
  inflating: 2/453576.jpg            
  inflating: 2/453577.jpg  

In [4]:
datasetPath = Path('/content/2')

imagenes = [p for p in datasetPath.rglob('*') ]

print(f"Total de imágenes: {len(imagenes)}")
print(f"Carpetas de primer nivel: {[f.name for f in sorted(datasetPath.iterdir()) if f.is_dir()][:10]}")

Total de imágenes: 21678
Carpetas de primer nivel: []


## 2. Comparativa de Detectores Faciales

Para evitar procesar el dataset completo con los tres detectores, vamos a comparar los tres detectores sobre una muestra aleatoria de 200 imágenes del dataset para evaluar su precisión visual y velocidad.

Los tres detectores que vamos a comparar son:

| Detector | Tipo | Velocidad | Precisión esperada |
|---|---|---|---|
| Haar Cascades | ML | Muy rápido | Baja  |
| HOG + SVM |  ML | Rápido | Media  |
| CNN (MMOD) |  Deep Learning | Lento  | Alta  |


### 2.1 Detector 1: Haar Cascades

In [12]:
# load the haar cascade face detector
print("[INFO] loading face detector...")
detectorHaar = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def detectarHaar(imagen_bgr):
    gray = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2GRAY)
    rects = detectorHaar.detectMultiScale(gray, scaleFactor=1.05,
        minNeighbors=5, minSize=(30, 30),
        flags=cv2.CASCADE_SCALE_IMAGE)
    return [(x, y, w, h) for (x, y, w, h) in rects] if len(rects) else []

[INFO] loading face detector...


### 2.2 Detector 2: dlib HOG + SVM

In [13]:
# auxiliary function from PyImageSearch helpers.py
def convert_and_trim_bb(image, rect):
    # extract the starting and ending (x, y)-coordinates of the bounding box
    startX = rect.left()
    startY = rect.top()
    endX = rect.right()
    endY = rect.bottom()

    # ensure the bounding box coordinates fall within the spatial
    # dimensions of the image
    startX = max(0, startX)
    startY = max(0, startY)
    endX = min(endX, image.shape[1])
    endY = min(endY, image.shape[0])

    # compute the width and height of the bounding box
    w = endX - startX
    h = endY - startY

    # return our bounding box coordinates
    return (startX, startY, w, h)

# load dlib's HOG + Linear SVM face detector
print("[INFO] loading HOG + Linear SVM face detector...")
detectorHOG = dlib.get_frontal_face_detector()

def detectarHOG(imagen_bgr):
    rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
    rects = detectorHOG(rgb, 1)
    return [convert_and_trim_bb(imagen_bgr, r) for r in rects]

[INFO] loading HOG + Linear SVM face detector...


### 2.3 Detector 3: dlib CNN (MMOD)

Descargamos el modelo preentrenado `mmod_human_face_detector.dat`.

In [7]:
!wget https://github.com/davisking/dlib-models/raw/master/mmod_human_face_detector.dat.bz2
!bunzip2 mmod_human_face_detector.dat.bz2

--2026-05-18 10:35:23--  https://github.com/davisking/dlib-models/raw/master/mmod_human_face_detector.dat.bz2
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/davisking/dlib-models/master/mmod_human_face_detector.dat.bz2 [following]
--2026-05-18 10:35:23--  https://raw.githubusercontent.com/davisking/dlib-models/master/mmod_human_face_detector.dat.bz2
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 694687 (678K) [application/octet-stream]
Saving to: ‘mmod_human_face_detector.dat.bz2’

mmod_human_face_det 100%[===================>] 678.41K  --.-KB/s    in 0.03s   

2026-05-18 10:35:24 (20.5 MB/s) 

In [14]:
print("[INFO] loading CNN face detector...")

cnn_model_path = Path('mmod_human_face_detector.dat')

detectorCNN = dlib.cnn_face_detection_model_v1(str(cnn_model_path))

def detectarCNN(imagen_bgr):
    rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
    results = detectorCNN(rgb, 1)
    return [convert_and_trim_bb(imagen_bgr, r.rect) for r in results]

[INFO] loading CNN face detector...


### 2.4 Visualización

Aplicamos los tres detectores sobre una muestra aleatoria de 200 imágenes y pintamos las bounding boxes de 3 imagenes de esta muestra.

In [15]:
random.seed(42)
muestra = random.sample(imagenes,200)

In [ ]:
detectores = {'Haar': detectarHaar, 'HOG': detectarHOG, 'CNN': detectarCNN}
colores = [(255,0,0), (0,200,0), (0,0,255)]

In [16]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for fila, ruta in enumerate(muestra[:3]):
    img = cv2.imread(str(ruta))
    for col, (nombre, fn) in enumerate(detectores.items()):
        copia = img.copy()
        caras = fn(copia)
        for (x, y, w, h) in caras:
            cv2.rectangle(copia, (x, y), (x+w, y+h), colores[col], 3)
        axes[fila][col].imshow(cv2.cvtColor(copia, cv2.COLOR_BGR2RGB))
        axes[fila][col].set_title(f"{nombre} — {len(caras)} cara(s)")
        axes[fila][col].axis('off')

plt.suptitle('Comparativa de detectores faciales')
plt.tight_layout()
plt.show()

<Figure size 1500x1200 with 9 Axes>

### 2.5 Evaluacion de rendimiento

Medimos el tiempo de procesado de cada detector sobre las 200 imágenes para tener datos del rendimiento.

In [17]:
imagenes_cargadas = [img for r in muestra[:200] if (img := cv2.imread(str(r))) is not None]
print(f"Imágenes cargadas: {len(imagenes_cargadas)}\n")

for nombre, fn in detectores.items():
    start = time.time()
    caras = sum(len(fn(img)) for img in imagenes_cargadas)
    elapsed = time.time() - start
    print(f"{nombre:<10} {elapsed:.2f}s — {caras} caras")

Imágenes cargadas: 200

Haar       72.38s — 579 caras
HOG        48.31s — 349 caras
CNN        19.72s — 347 caras


### 2.6 Elección del detector final: dlib CNN

| Detector | Tiempo | Caras detectadas |
|---|---|---|
| Haar Cascades | 72.38s | 579 |
| HOG  | 48.31s | 349 |
| CNN  | 19.72s | 347 |

La comparativa visual sobre las 3 imagenes de muestra complementa los números obtenidos, y nos lleva a las siguientes conclusiones:

- Haar tiene el mayor recall, vemos que en la foto de la piscina detecta 4 caras de niños con bounding boxes bastante acertadas, y en la foto de la reunión las 2 cajas principales están bien centradas. Sin embargo incluye falsos positivos claros en ambas fotos, lo que haria que tengamos recortes inútiles para el Notebook 3.
- HOG queda por detrás de los otros dos ya detecta menos caras que Haar sin la ventaja de precisión de CNN.
- CNN no detecta las caras pequeñas de la piscina, pero donde detecta sus bounding boxes son precisas y sin falsos positivos. Además, con GPU de colab es el más rápido de los tres con diferencia.

Elegimos CNN ya que una cara no detectada contamina menos los embeddings del Notebook 3 que un falso positivo con bounding box imprecisa, que le pasaría al modelo de reconocimiento facial un recorte que no representa una cara.Es por estas razones que lo elegimos como detector principal.

## 3. Normalización de Caras

Como vimos en el notebook 1 es necesario normalizarlas a un tamaño fijo antes de guardarlas. Los modelos de reconocimiento facial esperan imágenes de tamaño fijo, por lo que elegimos 160×160 px como resolución de entrada.

En el proceso de normalización por cara lo que haremos es primeramente añadir un margen de 0.2 para incluir toda la cara bien, recortamos la region donde esta esa cara y la redimencionamos a 160x160 px.

In [18]:
def recortarYNormalizar(imagen_bgr, x, y, w, h):
    ih, iw = imagen_bgr.shape[:2]
    margen_px_w = int(w * 0.20)
    margen_px_h = int(h * 0.20)
    x1 = max(0, x - margen_px_w)
    y1 = max(0, y - margen_px_h)
    x2 = min(iw, x + w + margen_px_w)
    y2 = min(ih, y + h + margen_px_h)
    if (x2 - x1) < 20 or (y2 - y1) < 20:
        return None
    recorte = imagen_bgr[y1:y2, x1:x2]
    recorte_norm = cv2.resize(recorte, (160, 160), interpolation=cv2.INTER_CUBIC)
    return recorte_norm

Verificamos visualmente el resultado de la normalización sobre las 10 primeras imágenes de la muestra en lugar de 3, ya que CNN es conservador y con solo 3 imágenes únicamente obteníamos 2 recortes, lo que resultaba poco para visualizar el resultado de la normalización.

In [21]:
todos_recortes = []

for ruta in muestra[:10]:
    img = cv2.imread(str(ruta))
    if img is None:
        continue
    for (x, y, w, h) in detectarCNN(img):
        recorte = recortarYNormalizar(img, x, y, w, h)
        if recorte is not None:
            todos_recortes.append(recorte)

print(f"Recortes obtenidos de la muestra: {len(todos_recortes)}")

Recortes obtenidos de la muestra: 18


In [23]:
n_mostrar = min(16, len(todos_recortes))
fig, axes = plt.subplots(1, n_mostrar, figsize=(3 * n_mostrar, 3))
for i, ax in enumerate(axes):
    ax.imshow(cv2.cvtColor(todos_recortes[i], cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.tight_layout()
plt.show()

<Figure size 4800x300 with 16 Axes>

In [ ]:
DRIVE_OUT = Path('/content/caras_recortadas')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print(f"Directorio de salida: {DRIVE_OUT}")